In [1]:
# 1. 必要なライブラリのインストール
!pip install umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
import umap
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定
OUT_DIR = "L2_Norm_Domination_Evaluation"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数）の6分割スケール
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

N_PER_GROUP = 100
N_SAMPLES = N_PER_GROUP * 2 # 合計200サンプル
results = []

print("=== L2ノルム支配による統計的評価（Silhouette Score）の錯覚の実証を開始します ===")

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(10, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行: {total_trials:,} (D={D_features})")

    np.random.seed(42 + step)

    # --- A. Normal Data (真の異質集団: 均質な分布で2つのクラスター) ---
    X_norm_A = np.random.uniform(0.1, 0.3, size=(N_PER_GROUP, D_features))
    X_norm_B = np.random.uniform(0.7, 0.9, size=(N_PER_GROUP, D_features))
    X_normal_raw = np.vstack([X_norm_A, X_norm_B])

    # --- B. Anomaly Data (少数のサンプルに極端な外れ値を注入) ---
    X_anomaly_raw = X_normal_raw.copy()
    # 全体の5%のサンプルに外れ値（L2ノルム破壊）を注入
    anomaly_indices = np.random.choice(N_SAMPLES, int(N_SAMPLES * 0.05), replace=False)
    for idx in anomaly_indices:
        anomaly_dim = np.random.randint(0, D_features)
        X_anomaly_raw[idx, anomaly_dim] *= 50.0 # 極端なスパイク

    # --- C. Skewed Data (頻度スキュー: 特定の次元が全体的に重い / Zipf則的) ---
    X_skewed_raw = X_normal_raw.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)
    X_skewed_raw[:, :hub_dims] *= skew_multiplier

    # 全て組成データ（simplex）へ正規化
    X_normal_simplex = X_normal_raw / np.sum(X_normal_raw, axis=1, keepdims=True)
    X_anomaly_simplex = X_anomaly_raw / np.sum(X_anomaly_raw, axis=1, keepdims=True)
    X_skewed_simplex = X_skewed_raw / np.sum(X_skewed_raw, axis=1, keepdims=True)

    # --- 距離行列の計算（Cosine） ---
    dist_norm = pairwise_distances(X_normal_simplex + 1e-10, metric='cosine')
    dist_anom = pairwise_distances(X_anomaly_simplex + 1e-10, metric='cosine')
    dist_skew = pairwise_distances(X_skewed_simplex + 1e-10, metric='cosine')

    # --- 評価関数: クラスタリングとSilhouette Scoreの計算 ---
    def evaluate_clustering(dist_matrix):
        # 階層クラスタリング（2クラスター想定）
        hc = AgglomerativeClustering(n_clusters=2, metric='precomputed', linkage='average')
        labels = hc.fit_predict(dist_matrix)
        # 距離行列に基づくシルエットスコア
        sil_score = silhouette_score(dist_matrix, labels, metric='precomputed')

        # UMAP
        reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
        emb = reducer.fit_transform(dist_matrix)

        return labels, sil_score, emb

    labels_norm, sil_norm, umap_norm = evaluate_clustering(dist_norm)
    labels_anom, sil_anom, umap_anom = evaluate_clustering(dist_anom)
    labels_skew, sil_skew, umap_skew = evaluate_clustering(dist_skew)

    # 記録
    results.append({
        'Step': step, 'Dimensions': D_features,
        'Silhouette_Normal': sil_norm,
        'Silhouette_Anomaly': sil_anom,
        'Silhouette_Skewed': sil_skew
    })

    # --- グラフ出力 ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Step {step}: L2 Norm Domination & Clustering Illusion (D={D_features})', fontsize=16)

    # 真のラベル（色分け用）
    y_true = np.array([0]*N_PER_GROUP + [1]*N_PER_GROUP)

    # 1. Normal
    axes[0].scatter(umap_norm[:, 0], umap_norm[:, 1], c=y_true, cmap='coolwarm', alpha=0.7)
    axes[0].set_title(f'Normal Data\nSilhouette: {sil_norm:.3f}')

    # 2. Anomaly
    axes[1].scatter(umap_anom[:, 0], umap_anom[:, 1], c=y_true, cmap='coolwarm', alpha=0.7)
    axes[1].set_title(f'Anomaly Injected (5% samples)\nSilhouette: {sil_anom:.3f}')
    # アノマリーサンプルの強調
    axes[1].scatter(umap_anom[anomaly_indices, 0], umap_anom[anomaly_indices, 1], facecolors='none', edgecolors='green', s=100, label='Anomaly')
    axes[1].legend()

    # 3. Skewed
    axes[2].scatter(umap_skew[:, 0], umap_skew[:, 1], c=y_true, cmap='coolwarm', alpha=0.7)
    axes[2].set_title(f'Frequency Skewed (Hubs)\nSilhouette: {sil_skew:.3f}')

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/l2_illusion_step_{step}.png")
    plt.close()

# 4. 集計とまとめグラフ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/l2_evaluation_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(trial_scales, df_results['Silhouette_Normal'], marker='o', color='blue', label='Normal (True Clusters)')
ax.plot(trial_scales, df_results['Silhouette_Anomaly'], marker='^', color='green', label='Anomaly (L2 Outliers)')
ax.plot(trial_scales, df_results['Silhouette_Skewed'], marker='s', color='red', label='Skewed (Hubs / Chimeras)')

ax.set_xscale('log')
ax.set_xlabel('Total Trials (log scale)')
ax.set_ylabel('Silhouette Score (Higher is visually "better")')
ax.set_title('The Illusion of Clustering Evaluation under Cosine Metric')

# アノマリーが「嘘の高得点」を出すことを強調
ax.annotate('Statistical Illusion:\nAnomalies artificially inflate score\nby destroying metric scale.',
            xy=(trial_scales[2], df_results['Silhouette_Anomaly'].iloc[2]),
            xytext=(trial_scales[2]*0.5, df_results['Silhouette_Anomaly'].iloc[2]-0.2),
            arrowprops=dict(facecolor='black', shrink=0.05),
            fontsize=10, bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="black", lw=1, alpha=0.8))

ax.grid(True, which="both", ls="--", alpha=0.5)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/summary_l2_illusion_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。")
files.download(f"{OUT_DIR}.zip")

=== L2ノルム支配による統計的評価（Silhouette Score）の錯覚の実証を開始します ===

[Step 1/6] 試行: 1,000 (D=10)

[Step 2/6] 試行: 10,000 (D=50)

[Step 3/6] 試行: 100,000 (D=500)

[Step 4/6] 試行: 1,000,000 (D=5000)

[Step 5/6] 試行: 5,000,000 (D=25000)

[Step 6/6] 試行: 10,000,000 (D=50000)

実験完了。全データを L2_Norm_Domination_Evaluation.zip に圧縮しました。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>